[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/06_%E9%87%8D%E5%9B%9E%E5%B8%B0%E5%88%86%E6%9E%90.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../ を有効化
    os.makedirs('../data', exist_ok=True)
    import numpy as np, pandas as pd
    D = '../data'; rng = np.random.default_rng(42)
    n = 120; cls = rng.choice(['A','B','C'], n)
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int)
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    days = pd.date_range('2026-06-01', periods=30)
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}
    won = np.array([rng.random()<wr[c] for c in ch])
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}
    rows = []
    for mo in months:
        for c in chs:
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:
        for c in (rng.random(size)<p): ab.append([grp,int(c)])
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計2級-06. 重回帰分析

複数の説明変数から1つの結果を予測・説明するのが**重回帰**。
2級頻出の、決定係数・偏回帰係数・回帰係数の検定・ダミー変数・多重共線性を学びます。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_csv('../data/students_scores.csv')
df.head()

## 1. 単回帰の復習 → 重回帰へ

`数学` を `英語`・`国語`・`勉強時間` から説明します。
$$ 数学 = b_0 + b_1\,英語 + b_2\,国語 + b_3\,勉強時間 $$

`statsmodels` を使うと、表計算ソフトのような回帰の出力が得られます。

In [ ]:
model = smf.ols('数学 ~ 英語 + 国語 + 勉強時間', data=df).fit()
print(model.summary())

## 2. 出力の読み方（2級の頻出ポイント）

- **coef（偏回帰係数）**：他の変数を一定にしたとき、その変数が1増えると数学が何点変わるか
- **R-squared（決定係数）**：当てはまりの良さ（0〜1、1に近いほど良い）
- **Adj. R-squared（自由度調整済み）**：変数の数を考慮した決定係数。変数比較に使う
- **P>|t|（回帰係数の検定）**：その変数が本当に効いているか（0.05未満で有意）

In [ ]:
print('決定係数 R^2      :', round(model.rsquared, 3))
print('自由度調整済み R^2:', round(model.rsquared_adj, 3))
print('\n偏回帰係数:')
print(model.params.round(3))
print('\n各係数のp値:')
print(model.pvalues.round(4))

## 3. 予測してみる

In [ ]:
new = pd.DataFrame({'英語':[70], '国語':[75], '勉強時間':[2.0]})
print('予測される数学の点数:', round(model.predict(new)[0], 1))

## 4. ダミー変数（質的データを回帰に入れる）

`クラス`(A/B/C)のような質的変数は、0/1の**ダミー変数**にして投入します。
`C(クラス)` と書くと自動でダミー化されます。

In [ ]:
model2 = smf.ols('数学 ~ 英語 + C(クラス)', data=df).fit()
print(model2.params.round(2))
print('→ [T.B],[T.C] は基準クラスAと比べた差を表す')

## 5. 多重共線性（マルチコ）

説明変数どうしが強く相関していると、係数が不安定になる問題。
**VIF**（分散拡大係数）で確認します。VIFが10を超えると要注意。

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = df[['英語', '国語', '勉強時間']].copy()
X['const'] = 1
vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    index=X.columns)
print(vif.round(2))
print('→ const以外が10未満なら多重共線性は問題なし')

---
## 🏆 練習問題

**問1.** `英語` を `数学`・`国語`・`勉強時間` で説明する重回帰を作り、
決定係数と、有意な（p<0.05）説明変数を答えよう。

**問2.** `sales_btob.csv` で `商談金額` を `業界`（ダミー）で説明する回帰を作り、
どの業界が金額を押し上げ／押し下げているか読み取ろう。

**問3.** 問1のモデルに`クラス`のダミーを加えると自由度調整済みR²は上がる？下がる？確かめよう。

In [ ]:
# 問1


In [ ]:
# 問2
btob = pd.read_csv('../data/sales_btob.csv')


<details><summary>解答例</summary>

```python
m = smf.ols('英語 ~ 数学 + 国語 + 勉強時間', data=df).fit()
print(m.rsquared, m.pvalues)
smf.ols('商談金額 ~ C(業界)', data=btob).fit().params
```
</details>